# 🌸 Iris Dataset Analysis & KNN Classification Model
---
This notebook loads the Iris dataset from online, performs exploratory data analysis (EDA), and builds a **K-Nearest Neighbors (KNN)** classification model.

In [ ]:
# ============================
# 📦 Import Libraries
# ============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print('All libraries imported successfully! ✅')

## 1️⃣ Load Dataset from Online

In [ ]:
# Online theke Iris dataset download
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
df = pd.read_csv(url)

print(f"Dataset Shape: {df.shape}")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
df.head(10)

## 2️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Dataset er basic info
print("=" * 50)
print("📋 Dataset Info:")
print("=" * 50)
df.info()
print("\n")

print("=" * 50)
print("📊 Statistical Summary:")
print("=" * 50)
df.describe()

In [ ]:
# Missing values check
print("=" * 50)
print("❓ Missing Values:")
print("=" * 50)
print(df.isnull().sum())
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

# Duplicate check
print(f"\n🔁 Duplicate Rows: {df.duplicated().sum()}")

# Class distribution
print("\n=" * 50)
print("🌼 Species Distribution:")
print("=" * 50)
print(df['species'].value_counts())

In [ ]:
# Data Visualization - Species Count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.countplot(x='species', data=df, palette='viridis', ax=axes[0])
axes[0].set_title('Species Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Species')
axes[0].set_ylabel('Count')

# Pie chart
df['species'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#2ecc71','#3498db','#e74c3c'], ax=axes[1])
axes[1].set_title('Species Percentage', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Pairplot - Feature relationships
sns.pairplot(df, hue='species', palette='Set2', diag_kind='kde')
plt.suptitle('Feature Pairplot', y=1.02, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot - Outlier detection
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for i, (feature, color) in enumerate(zip(features, colors)):
    ax = axes[i // 2][i % 2]
    sns.boxplot(x='species', y=feature, data=df, palette='Set2', ax=ax)
    ax.set_title(f'{feature} by Species', fontsize=12, fontweight='bold')

plt.suptitle('Boxplot - Outlier Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3️⃣ Data Preprocessing

In [ ]:
# Features (X) and Target (y) separate kora
X = df.drop('species', axis=1)
y = df['species']

# Label Encoding - species ke number e convert
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print("Classes:", le.classes_)
print("Encoded:", np.unique(y_encoded))

# Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"\n📊 Training Set: {X_train.shape[0]} samples")
print(f"📊 Testing Set:  {X_test.shape[0]} samples")

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Data preprocessing complete!")

## 4️⃣ Finding Best K Value

In [ ]:
# Best K value khunje ber kora (1-25)
k_range = range(1, 26)
k_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    score = knn.score(X_test_scaled, y_test)
    k_scores.append(score)
    
# Best K
best_k = list(k_range)[np.argmax(k_scores)]
best_score = max(k_scores)
print(f"🏆 Best K = {best_k} with Accuracy = {best_score:.4f}")

# Plot K vs Accuracy
plt.figure(figsize=(10, 5))
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K = {best_k}')
plt.xlabel('K Value', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('K vs Accuracy (Elbow Method)', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5️⃣ KNN Model Training & Evaluation

In [ ]:
# Best K diye final KNN model
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)

# Prediction
y_pred = knn_model.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("=" * 50)
print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")
print("=" * 50)

# Classification Report
print("\n📋 Classification Report:")
print("-" * 50)
target_names = le.classes_
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
# Confusion Matrix Visualization
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title(f'Confusion Matrix (K={best_k})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6️⃣ New Data Prediction (Example)

In [ ]:
# Notun data diye prediction test
new_data = np.array([[5.1, 3.5, 1.4, 0.2],   # Expected: setosa
                     [6.7, 3.0, 5.2, 2.3],   # Expected: virginica
                     [5.9, 3.0, 4.2, 1.5]])  # Expected: versicolor

new_data_scaled = scaler.transform(new_data)
predictions = knn_model.predict(new_data_scaled)
predicted_species = le.inverse_transform(predictions)

print("=" * 50)
print("🔮 New Data Predictions:")
print("=" * 50)
for i, (data, species) in enumerate(zip(new_data, predicted_species)):
    print(f"  Sample {i+1}: {data} --> 🌸 {species}")

print("\n✅ KNN Model successfully built and tested!")